# ERSP Analysis
Raw signals are loaded from network storage across three patient groups using a single format-agnostic loader that handles TRC, EDF, and H5 files (load_first_raw_in_dir). Non-neural channels are removed (filter_aux_channels), and for EL patients only electrodes with anatomical labels are kept. Signals are then rereferenced to the average of white matter contacts defined per patient (apply_wm_reref), and line noise is removed at each harmonic only if a real peak is detected, with notch strength set automatically (notch_mains_harmonics). Trial timing comes from photodiode triggers saved as TSV files per patient, and trials are kept only if the stimulus lasted at least 0.5s and the response no more than 10s, with IQR used to remove remaining outliers (collect_trials). ERSPs are computed using short-time Fourier transform and warped so each trial is split 50/50 between stimulus and post-stimulus, baseline corrected before stimulus onset (compute_ersp). White matter channels are skipped. Outputs per channel are an ERSP plot, a high-gamma heatmap sorted by trial duration (plot_hg_trials), and for clustering a raw matrix and a clean image. QC outputs are two PSDs (before and after processing) and a full recording montage with trial markers (plot_montage_overview).

For each patient, loads and preprocesses raw neural signals, then runs one or both of two parallel pipelines controlled by boolean flags.

## Processing Steps (shared for all patients)

#### 1. Data Loading
- Builds patient-specific paths (raw + prep directories)
- Loads raw signals using format-agnostic loader (TRC/EDF/H5)
- Removes auxiliary channels (ECG, DC, markers etc.)

#### 2. Channel Filtering (EL patients only)
- **SEEG patients**: keeps only channels with `_` in name (e.g. `A_L6`)
- **Grid patients** (e.g. EL044): keeps only channels matching defined prefixes with a digit (e.g. `Pa1`, `T17`, `postP3`)
- Skips patient entirely if no neural channels remain

#### 3. Preprocessing
- Saves **PSD before processing** (if flag on)
- Applies **white matter rereferencing**
- Applies **adaptive mains notch filtering**
- Saves **PSD after processing** (if flag on)

#### 4. Trial Collection
- Reads trial TSV files from `prep0`
- Applies hard duration filters: `min_stim_s=0.5`, `max_post_s=10`
- Trims outliers using IQR method
- Saves QC report and histogram


## Pipeline A — ERSP Pipeline
*Runs if `RUN_ERSP_PIPELINE=True`*

- Saves **montage overview plot** with trial onset/offset markers
- For each condition and channel:
  - Computes **ERSP** (time-frequency power map)
  - Saves **ERSP plot** (`.tif`)
  - Saves **HG trials plot** (high-gamma, trial-by-trial heatmap)

## Pipeline B — Cluster Export
*Runs if `RUN_CLUSTER_EXPORT=True`*

- Skips non-neural, bad, and WM channels
- For each condition and channel:
  - Reuses ERSP result if Pipeline A also ran (no recomputation)
  - Saves **ERSP matrix** (`.npy`) for clustering input
  - Saves **clean ERSP image** (`.png`) for clustering input

## Outputs
| Product | Location | Pipeline |
|---|---|---|
| PSD_raw | `outputs/04_ersp_LM/<pid>/LM/PSD_raw` | A |
| PSD_clean | `outputs/04_ersp_LM/<pid>/LM/PSD_clean` | A |
| Report + montage | `outputs/04_ersp_LM/<pid>/LM/Report` | A |
| ERSP plots | `outputs/04_ersp_LM/<pid>/LM/ERSP/<cond>` | A |
| HG plots | `outputs/04_ersp_LM/<pid>/LM/HG/<cond>` | A |
| ERSP matrix | `outputs/04_ersp_LM_RAWONLY/<pid>/LM/ERSP_matrix/<cond>` | B |
| ERSP clean PNG | `outputs/04_ersp_LM_RAWONLY/<pid>/LM/ERSP_clean/<cond>` | B |

## Imports & run controls 
(the only place you change things is here and cell 4)


In [1]:
# ============================================================
# 140_ERSP_analysis_pipeline.ipynb
# Cell 1 — Imports & run controls
# ============================================================
import os, glob, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from itertools import groupby
import csv

from functions import lf_io_utils as io, lf_trials as tr, lf_ersp as fe, config as cfg
from functions.config import (PAT_PATIENTS, PAT_PRESETS, EL_PATIENTS, EL_PRESETS,
                               MICROEPI_MAT_PATIENTS, MICROEPI_MAT_PRESETS, COND_ALIAS)
import LFfunctions_PDextract as LF
from LF_pd import load_patient_raw

# ------------------------------------------------------------------
# RUN CONTROLS  ← edit these before each run
# ------------------------------------------------------------------
BLOCK = "LM"

# Toggle sections
RUN_PD_EXTRACTION    = True
RUN_ERSP_PIPELINE    = True
RUN_CLUSTER_EXPORT   = True
DO_MONTAGE_PSD_PLOTS = True

# Output roots
ERSP_SCRIPT_NAME    = "04_ersp_LM"
RAWONLY_SCRIPT_NAME = "04_ersp_LM_RAWONLY"
run_root_ersp       = os.path.join(cfg.outputs_root, ERSP_SCRIPT_NAME)
run_root_raw        = os.path.join(cfg.outputs_root, RAWONLY_SCRIPT_NAME)

# ERSP params (from config.py)
ersp_params = fe.ERSPParams(
    nperseg=cfg.nperseg, nfft=cfg.nfft, noverlap=cfg.noverlap,
    baseline_w=cfg.baseline_w, proportions=cfg.proportions,
    n_time_bins=cfg.n_time_bins, vmin=cfg.vmin, vmax=cfg.vmax, fmax=cfg.fmax
)

print("Controls loaded.")

Controls loaded.


In [2]:
# ============================================================
# Cell 2 — Helper functions
# (implementations live in lf_ersp.py and lf_io_utils.py)
# ============================================================

# Direct aliases
notch_mains_harmonics = fe.notch_mains_harmonics
fill_nans_nearest     = fe.fill_nans_nearest
save_clean_png        = fe.save_clean_png
plot_psd_overview     = fe.plot_psd_overview
_is_non_neural        = io._is_non_neural
_ensure               = io.ensure_dir

# MicroEPI .mat helpers (used for G-04 / G-05 / G-06 — saved as PAT_<bids_num>)
import lf_micromacro as mm

# Thin cfg-binding wrappers (keep pipeline cells unchanged)
def apply_notch_with_audit(signals, fs, patient_id, pid_raw):
    return fe.apply_notch_with_audit(
        signals, fs, patient_id, pid_raw,
        notch_patients=getattr(cfg, "notch_patients", []),
        mains_base=getattr(cfg, "mains_base", 50.0),
        fmax=getattr(cfg, "fmax", 500.0),
        repeats=getattr(cfg, "notch_repeats", 1),
        peak_z_thresh=getattr(cfg, "notch_peak_z_thresh", 3.0),
    )

def apply_wm_reref(signals, names, patient_id, *, electrodes_tsv_pattern=None):
    """Returns (signals, reref_label, wm_skip_set).

    `electrodes_tsv_pattern` overrides the auto-resolved BIDS path — used for
    MicroEPI .mat patients whose TSV lives outside the standard cohort layout.

    Raises ValueError if cfg.reref_type is not 'WM', if no WM channels are
    found for the patient, or if WM rereferencing ultimately failed to apply.
    All outputs in this pipeline MUST be WM rereferenced.
    """
    if str(cfg.reref_type).upper() != "WM":
        raise ValueError(
            f"[reref] cfg.reref_type is '{cfg.reref_type}' — only 'WM' is "
            f"allowed in this pipeline. Update config.py and re-run."
        )
    wm_idx = io.wm_indices_for_patient(patient_id, names,
                                        electrodes_tsv_pattern=electrodes_tsv_pattern)
    if not wm_idx:
        raise ValueError(
            f"[reref] {patient_id}: no WM channels found — cannot apply WM "
            f"rereferencing. Check the electrodes TSV and WM threshold."
        )
    bad = getattr(cfg, "bad_channels_manual", {}).get(patient_id, [])
    signals_r, used, excluded = fe.apply_wm_reference_with_exclusions(
        signals, names, wm_idx, bad)
    if not used:
        raise ValueError(
            f"[reref] {patient_id}: WM channels were found but none were used "
            f"after exclusions — cannot guarantee WM rereferencing. "
            f"Check bad_channels_manual and available WM channels."
        )
    return signals_r, "WM", set(used) | set(excluded)


def _load_signals_and_prep_for_patient(pid_raw):
    """Route a patient to its loader.

    For MicroEPI .mat patients (G-04/G-05/G-06): load only the macro signals
    and use the BIDS electrodes.tsv from `MICROEPI_MAT_PRESETS`. Everything
    else falls through to the standard TRC/EDF/H5 loader.

    Returns
    -------
    patient_id              : str (PAT_<num> for MicroEPI .mat patients)
    signals, names, fs      : as returned by the chosen loader
    prep_dir                : where collect_trials should read from
    electrodes_tsv_pattern  : explicit pattern for WM reref (or None)
    """
    pid_str = str(pid_raw)
    if pid_str in getattr(cfg, "MICROEPI_MAT_PATIENTS", []):
        preset = cfg.MICROEPI_MAT_PRESETS[pid_str]
        patient_id = preset["pat_name"]
        signals, names, fs, _pd = mm.load_microepi_macros_for_pipeline(
            pid_str, cfg.MICROEPI_MAT_PRESETS)
        prep_dir = os.path.join(os.path.dirname(preset["data_dir"]), "prep0")
        electrodes_tsv = preset["electrodes_tsv"]
        print(f"  [paths] raw_dir : {preset['data_dir']}")
        print(f"  [paths] prep_dir: {prep_dir}")
    else:
        patient_id, raw_dir, prep_dir = io.build_paths_for_patient(pid_raw, cfg.block_name)
        signals, names, fs = io.load_first_raw_in_dir(raw_dir)
        electrodes_tsv = None  # auto-resolve from cohort
    return patient_id, signals, names, fs, prep_dir, electrodes_tsv


print("Helpers loaded.")


Helpers loaded.


## Part 1 — Photodiode / trial extraction
Runs `LF_pd.load_patient_raw` and `LFfunctions_PDextract` for each patient in `PD_PATIENTS`.
Output: TSV timing files saved to each patient's `prep0` folder.
Set `RUN_PD_EXTRACTION = False` in Cell 1 to skip.

### PD extraction loop (PAT / EL / MicroEPI unified, calls LF_pd as-is)


In [3]:
if not RUN_PD_EXTRACTION:
    print("[skip] PD extraction (RUN_PD_EXTRACTION=False)")
else:
    # Active patients (empty list = skip that group)
    PAT_PATIENTS      = []
    EL_PATIENTS       = []#"EL042","EL043","EL044","EL045"]
    MICROEPI_MAT_PATIENTS = []

    all_patients = (
        [(pid, "PAT",      PAT_PRESETS)      for pid in PAT_PATIENTS] +
        [(pid, "EL",       EL_PRESETS)       for pid in EL_PATIENTS] +
        [(pid, "MICROEPI", MICROEPI_MAT_PRESETS) for pid in MICROEPI_MAT_PATIENTS]
    )

    for pid, group, presets in all_patients:
        preset = presets.get(pid)
        if preset is None:
            print(f"[skip] {pid}: no preset"); continue

        patient_id = f"PAT_{pid}" if group == "PAT" else f"MicroEPI-{pid}" if group == "MICROEPI" else pid
        print(f"\n=== {patient_id} ===")

        try:
            if group == "PAT":
                base_path = fr"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\{patient_id}\task_FBM\data_{BLOCK}\raw"
                raw_signals, channel_names, sampling_rate = io.load_trc_and_signals(glob.glob(os.path.join(base_path, "*.TRC"))[0])
                save_path = os.path.join(os.path.dirname(base_path), "prep0")

            elif group == "EL":
                info          = load_patient_raw(pid, block_name=BLOCK, use_el_mat_fallback=False, verbose=True)
                raw_signals   = info["raw_signals"]
                sampling_rate = info["sampling_rate"]
                channel_names = list(info["channel_names"])
                save_path     = info["save_path"]
                exp_file      = info["matching_files_onsets"][0] if info["matching_files_onsets"] else None

            elif group == "MICROEPI":
                base_path = fr"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\MICROEPI\{patient_id}\task_FBM\data_{BLOCK}\raw"
                buckets   = _load_microepi_folder(base_path)
                if not buckets: raise RuntimeError("No .mat files found")
                trig_key  = preset["trig"].lower()
                kind      = next((k for k in ('micro','macro') if k in buckets and any(c.lower()==trig_key for c in buckets[k][2])), list(buckets.keys())[0])
                raw_signals, sampling_rate, channel_names = buckets[kind][0], buckets[kind][1], list(buckets[kind][2])
                save_path = os.path.join(os.path.dirname(base_path), "prep0")

            if group != "EL":
                exp_files = glob.glob(os.path.join(base_path, "*.tsv")) or glob.glob(os.path.join(base_path, "*.txt"))
                exp_file  = exp_files[0] if exp_files else None

            lower_map = {str(c).lower(): str(c) for c in channel_names}
            trig_key  = preset["trig"].lower()
            if trig_key not in lower_map:
                print(f"  [warn] trigger '{preset['trig']}' not found — skipping"); continue
            pd_name = lower_map[trig_key]

            mt = preset["manual_trig"]
            manual_path = None
            if mt:
                manual_path = mt if os.path.isabs(mt) else os.path.join(save_path, mt)
                if not os.path.exists(manual_path):
                    print(f"  [warn] manual triggers not found: {manual_path}")
                    manual_path = None

            on_abs, off_abs, metrics = LF.get_trigger_indexes_photodiode(
                raw_signals=raw_signals, sampling_rate=sampling_rate,
                channel_names=channel_names, trig_name=pd_name,
                time_range=preset["time_range"], threshold_val=0.40,
                flip_trigs=preset["flip"],
                trial_ids=preset["trial_ids"], invalid_trials=preset["invalid_trials"],
                ignore_invalid=False, fake_trials=preset["fake_trials"],
                extra_table_path=exp_file, manual_trigs_path=manual_path,
                return_extra_metrics=True,
            )
            print(f"  Paired trials: {len(on_abs)}")
            LF.parse_and_save(pid, patient_id, on_abs, off_abs, metrics,
                              sampling_rate, save_path, BLOCK, exp_file,
                              preset["trial_ids"], preset["trig"],
                              cond_alias=COND_ALIAS)

        except Exception as e:
            print(f"[error] {patient_id}: {e}")

    # --- MicroEPI .mat pipeline (G-04, G-05, G-06) ---
    # Photodiode lives inside the .mat; events TSV alongside it.
    # Prep0 is written next to data_dir so collect_trials can find it later.
    for pid in getattr(cfg, "MICROEPI_MAT_PATIENTS", []):
        print("")
        print("-.-*"*20)
        try:
            preset   = cfg.MICROEPI_MAT_PRESETS[pid]
            prep_dir = os.path.join(os.path.dirname(preset["data_dir"]), "prep0")
            mm.extract_microepi_pd_to_prep0(pid, cfg.MICROEPI_MAT_PRESETS, prep_dir)
            print(f"  [{preset['pat_name']}] PD extraction done -> {prep_dir}")
        except Exception as e:
            print(f"[error] MicroEPI {pid}: {e}")

    print("\n[PD extraction done]")



[PD extraction done]


## Part 2 — ERSP pipeline + cluster export
Reads raw data and `prep0` TSVs. Toggles:
- `RUN_ERSP_PIPELINE = True` → ERSP plots, HG plots, Report, montage QC into `04_ersp_LM/`
- `RUN_CLUSTER_EXPORT = True` → per-channel `ERSP_matrix/*.npy` and `ERSP_clean/*.png` into `04_ersp_LM_RAWONLY/` (consumed by `02_FBM_Clustering`)

Both flags can be on simultaneously — the loop computes the ERSP once per (channel, condition) and dispatches outputs to whichever pipeline is enabled.

In [4]:
cfg.patient_ids#=['3415']
    

['EL036', 'EL038', 'EL039']

In [ ]:
import gc, os
import pandas as pd

if not RUN_ERSP_PIPELINE and not RUN_CLUSTER_EXPORT:
    print("[skip] both pipelines disabled")
else:
    # Per-patient summary collected during the loop and printed/saved at the end.
    # Status codes: ok | ok-no-trials | error-load | error-reref | error-other
    wm_report_rows = []

    def process_patient(pid_raw):
        """
        Process one patient end-to-end. Wrapped in a function so all heavy
        intermediates (raw signals, ERSP cubes, matplotlib figures) become
        garbage-collectable on return — keeps RAM bounded across 16+ patients
        without needing a kernel restart.
        """
        report = {
            "pid_raw": str(pid_raw),
            "patient_id": "",
            "status": "",
            "n_channels_in": 0,
            "n_channels_neural": 0,
            "n_channels_unknown_dropped": 0,
            "n_channels_used": 0,
            "n_wm_used": 0,
            "wm_channels_used": "",
            "wm_channels_excluded_as_bad": "",
            "error": "",
        }
        try:
            patient_id, signals, names, fs, prep_dir, _wm_tsv = _load_signals_and_prep_for_patient(pid_raw)
            report["patient_id"] = patient_id
            report["n_channels_in"] = len(names)

            signals, names, *_ = io.filter_aux_channels(signals, names)
            if str(pid_raw).startswith("EL"):
                if pid_raw in cfg.EL_GRID_PATIENTS:
                    prefixes = cfg.EL_GRID_KEEP_PREFIXES.get(pid_raw, ())
                    keep = [i for i, nm in enumerate(names)
                            if any(str(nm).startswith(p) for p in prefixes)
                            and any(c.isdigit() for c in str(nm))]
                else:
                    keep = [i for i, nm in enumerate(names) if "_" in str(nm)]
                signals = signals[:, keep]
                names   = [names[i] for i in keep]
                if len(names) == 0:
                    print(f"[skip] {patient_id}: no neural channels")
                    report["status"] = "no-neural-channels"
                    return report
            report["n_channels_neural"] = len(names)

            # === PER-PATIENT CHANNEL-NAME OVERRIDE (EL043 etc.) ===
            # Patients in cfg.STRIP_HEMI_PATIENTS have raw EDF channel names
            # like 'A_L1' where the hemisphere letter sits between the
            # electrode and number, but their fsaverage coord CSV uses the
            # bare convention 'A1' (with hemi tracked in its own column).
            # Strip the _L# / _R# infix here so:
            #   * the WM marker matches TSV 'A5' against EDF 'A_L5' → 'A5'
            #   * ERSP filenames become 'EL043_..._ERSP_A1_TN.tif'
            #   * labels.csv 'electrode' column ends up as 'A1, A2, ...'
            #   * 252's contact-name merge joins A1 ↔ coord A1
            # Other EL patients (EL037/38/40/45) are NOT in this set — their
            # coord CSVs already bake the hemisphere into the name (AL1,
            # aHR1, OFR1, AR1), so the current pass-through behaviour is
            # already correct for them.
            if patient_id in getattr(cfg, "STRIP_HEMI_PATIENTS", set()):
                import re as _re
                _hemi_re = _re.compile(r"_(?:[LR])(?=\d)")
                names = [_hemi_re.sub("", str(nm)) for nm in names]
                print(f"  [{patient_id}] stripped _L#/_R# per cfg.STRIP_HEMI_PATIENTS  "
                      f"-> first few: {names[:8]}")

            # === DROP "UNKNOWN" PARCELLATION CHANNELS ===
            # Reads the same electrodes TSV as the WM derivation. Channels whose
            # tissueLabel is empty or starts with "Unknown" are skipped entirely
            # — no ERSP computed, no .npy saved, never seen by clustering.
            
            
            
            try:
                unk_idx = set(io.unknown_indices_for_patient(
                    patient_id, names, electrodes_tsv_pattern=_wm_tsv,
                ))
            except Exception as _e_unk:
                print(f"[warn] {patient_id}: Unknown-channel lookup failed ({_e_unk}); keeping all")
                unk_idx = set()
            # Protect surface ECoG contacts from the Unknown drop for mixed
            # grid+depth patients. BIDS only parcellates channels that pierce
            # cortex, so grid/strip contacts always come back as Unknown —
            # With this — protect grid prefixes for mixed patients BEFORE the drop:
            if patient_id in getattr(cfg, "MIXED_GRID_DEPTH_PATIENTS", set()):
                keep_prefixes = cfg.MIXED_GRID_KEEP_PREFIXES.get(patient_id, ())
                if keep_prefixes:
                    protected = {i for i in unk_idx
                                 if any(str(names[i]).startswith(p) for p in keep_prefixes)}
                    if protected:
                        print(f"  [{patient_id}] protected {len(protected)} grid channels "
                              f"from Unknown drop (prefixes={keep_prefixes})")
                    unk_idx -= protected

            if unk_idx:
                report["n_channels_unknown_dropped"] = len(unk_idx)
                keep = [i for i in range(len(names)) if i not in unk_idx]
                signals = signals[:, keep]
                names   = [names[i] for i in keep]
                print(f"  [{patient_id}] dropped {len(unk_idx)} 'Unknown' channels (no parcellation in TSV)")

            # WM rereferencing. Grid patients (cfg.EL_GRID_PATIENTS) without WM
            # contacts fall through with no rereferencing — apply_wm_reref would
            # otherwise raise. Non-grid patients still fail loud since they're
            # expected to have WM channels.
            is_grid    = str(pid_raw) in getattr(cfg, "EL_GRID_PATIENTS", set())
            wm_all     = io.wm_labels_for_patient(patient_id, electrodes_tsv_pattern=_wm_tsv)
            _bad_norm_for_report = {io.normalize_label(b)
                                    for b in getattr(cfg, "bad_channels_manual", {}).get(patient_id, [])}
            _name_norm = [io.normalize_label(n) for n in names]
            wm_in_sig  = [n for n in wm_all if n in _name_norm]
            wm_usable  = [n for n in wm_in_sig if n not in _bad_norm_for_report]

            if wm_usable:
                # Standard path: at least one usable WM channel.
                try:
                    signals, reref, wm_skip = apply_wm_reref(
                        signals, names, patient_id, electrodes_tsv_pattern=_wm_tsv,
                    )
                    wm_excluded_norm = [n for n in wm_in_sig if n in _bad_norm_for_report]
                    report["n_wm_used"]                    = len(wm_usable)
                    report["wm_channels_used"]             = "|".join(sorted(wm_usable))
                    report["wm_channels_excluded_as_bad"]  = "|".join(sorted(wm_excluded_norm))
                except ValueError as _e_reref:
                    print(f"[error] {patient_id}: WM reref failed — {_e_reref}")
                    report["status"] = "error-reref"
                    report["error"]  = str(_e_reref)
                    return report
            elif is_grid:
                # Grid patient without WM contacts (e.g. EL044): pass signals through
                # unchanged and tag the reref as 'NONE' so the .npy filenames are
                # parseable by s10_load_ersps (which already accepts a 'None' tag
                # in the middle slot — see its docstring example).
                print(f"[note] {patient_id}: grid patient with no WM contacts — "
                      f"falling back to reref='NONE' (no rereferencing applied)")
                reref   = "NONE"
                wm_skip = set()
                report["n_wm_used"]                   = 0
                report["wm_channels_used"]            = ""
                report["wm_channels_excluded_as_bad"] = ""
                report["status"]                      = "ok-no-wm-grid"   # overwritten to 'ok' at end if pipeline completes
            else:
                # Non-grid patient with no usable WM → fail loud (preserves prior behaviour).
                msg = f"no WM channels available for {patient_id} (not a grid patient)"
                print(f"[error] {msg}")
                report["status"] = "error-reref"
                report["error"]  = msg
                return report

            signals = apply_notch_with_audit(signals, fs, patient_id, pid_raw)

            if RUN_ERSP_PIPELINE and DO_MONTAGE_PSD_PLOTS:
                fe.plot_psd_overview(
                    signals=signals, fs=fs, names=names,
                    save_root=io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "PSD_clean"),
                    patient_id=patient_id, block_name=cfg.block_name,
                    fmax=cfg.fmax, mains_base=getattr(cfg, "mains_base", 50.0), dpi=600,
                )

            report_dir  = io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "Report")
            report_path = os.path.join(report_dir, f"{patient_id}_IQR.tsv")
            cond_groups = tr.collect_trials(prep_dir, fs, outlier_method="IQR",
                                            iqr_k=cfg.iqr_k, report_path=report_path,
                                            patient_id=patient_id, max_post_s=cfg.max_post_s)
            if not cond_groups:
                print(f"[skip] {patient_id}: no trials")
                report["status"] = "ok-no-trials"
                return report

            if RUN_ERSP_PIPELINE:
                tr.plot_montage_overview(
                    signals=signals, fs=fs, names=names,
                    cond_groups=cond_groups, save_dir=report_dir, patient_id=patient_id,
                )
                ersp_root = io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "ERSP")
                hg_root   = io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "HG")

            if RUN_CLUSTER_EXPORT:
                mat_root = _ensure(io.patient_output_dir(run_root_raw, patient_id, cfg.block_name, "ERSP_matrix"))
                img_root = _ensure(io.patient_output_dir(run_root_raw, patient_id, cfg.block_name, "ERSP_clean"))
                _bad_norm = {io.normalize_label(b)
                             for b in getattr(cfg, "bad_channels_manual", {}).get(patient_id, [])}
                skip      = set(n for n in names if _is_non_neural(n))
                skip     |= {n for n in names if io.normalize_label(n) in _bad_norm}
                skip     |= wm_skip

            for cond, (onsets, offsets, trial_ends) in cond_groups.items():
                if RUN_ERSP_PIPELINE:
                    ersp_dir = _ensure(os.path.join(ersp_root, cond))
                    hg_dir   = _ensure(os.path.join(hg_root, cond))
                if RUN_CLUSTER_EXPORT:
                    out_mat  = _ensure(os.path.join(mat_root, cond))
                    out_png  = _ensure(os.path.join(img_root, cond))

                print(f"  {cond}: ", end="", flush=True)
                for ci, chan_name in enumerate(names):
                    if chan_name in wm_skip: continue
                    if RUN_CLUSTER_EXPORT and chan_name in skip and not RUN_ERSP_PIPELINE: continue

                    res = fe.compute_ersp(
                        signals=signals, fs=fs, onsets=onsets, offsets=offsets, channel_idx=ci,
                        trial_ends=trial_ends, mode=cfg.mode, time_window=cfg.time_window,
                        params=ersp_params,
                    )

                    if RUN_ERSP_PIPELINE:
                        fe.plot_ersp(res, patient_id=patient_id, condition=cond,
                                     reref_type=reref, chan_name=chan_name,
                                     save_dir=ersp_dir, params=ersp_params,
                                     plot_title=False, save_sidecar=False)
                        fe.plot_hg_trials(
                            signals=signals, fs=fs, onsets=onsets, offsets=offsets, channel_idx=ci,
                            chan_name=chan_name, patient_id=patient_id, condition=cond, reref_type=reref,
                            time_window=cfg.time_window, baseline_w=cfg.baseline_w,
                            hg_band=cfg.hg_band, smooth_ms=cfg.hg_smooth_ms,
                            vmin=cfg.hg_vmin, vmax=cfg.hg_vmax,
                            save_dir=hg_dir, add_separators=False, sort_ascending=True,
                            trial_end_indices=trial_ends, sort_by="stim",
                        )

                    if RUN_CLUSTER_EXPORT and chan_name not in skip:
                        A = np.array(res["avg_db"], float)
                        if np.isnan(A).any():
                            print(f"[warn] {patient_id} {cond} {chan_name}: {int(np.isnan(A).sum())} NaNs → filling")
                            fill_nans_nearest(A)
                        mode_tag = "_TN" if str(res["meta"]["mode"]).upper() == "TN" else ""
                        stem = f"{patient_id}_{cond}_{reref}_ERSP_{chan_name}{mode_tag}"
                        np.save(os.path.join(out_mat, f"{stem}.npy"), A)
                        save_clean_png(A, vmin=ersp_params.vmin, vmax=ersp_params.vmax,
                                       path_png=os.path.join(out_png, f"{stem}_CLEAN.png"))
                        del A
                    del res
                print(" done")

            # Preserve the "ok-no-wm-grid" tag if we set it earlier;
            # otherwise this is a fully-normal "ok".
            if not report["status"]:
                report["status"] = "ok"
            return report
        except Exception as e:
            print(f"[error] {pid_raw}: {e}")
            report["status"] = report["status"] or "error-other"
            report["error"]  = str(e)
            return report

    # ────────────────────────────────────────────────────────────────────────
    # Main loop. Per-patient bodies run inside process_patient(); we then
    # explicitly close all matplotlib figures + GC to keep memory bounded.
    # ────────────────────────────────────────────────────────────────────────
    for pid_raw in cfg.patient_ids:
        print("\n")
        print(pid_raw)
        row = process_patient(pid_raw)
        wm_report_rows.append(row)
        plt.close("all")
        gc.collect()

    print("\nPipeline done.")

    # ────────────────────────────────────────────────────────────────────────
    # End-of-batch WM REREFERENCING REPORT
    # Printed to the notebook for inspection, plus saved as a TSV under
    # outputs/04_ersp_LM_RAWONLY/wm_reref_report.tsv (survives kernel restart).
    # ────────────────────────────────────────────────────────────────────────
    df_report = pd.DataFrame(wm_report_rows)
    print("\n" + "=" * 72)
    print("WM REREFERENCING REPORT (per patient)")
    print("=" * 72)
    if len(df_report) == 0:
        print("(no patients processed)")
    else:
        # Compact print: status | n_wm_used | n_channels_used | error?
        for _, r in df_report.iterrows():
            status_tag = {
                "ok":                  "  OK   ",
                "ok-no-trials":        "OK-NOTR",
                "ok-no-wm-grid":       "OK-GRID",
                "no-neural-channels":  "NO-CHAN",
                "error-reref":         "REREF✗ ",
                "error-load":          "LOAD✗  ",
                "error-other":         "ERR✗   ",
                "":                    "??     ",
            }.get(r["status"], r["status"])
            line = (f"  {status_tag}  {r['patient_id']:<14}  "
                    f"in={r['n_channels_in']:>4}  "
                    f"neural={r['n_channels_neural']:>4}  "
                    f"unknown_dropped={r['n_channels_unknown_dropped']:>3}  "
                    f"used={r['n_channels_used']:>4}  "
                    f"wm={r['n_wm_used']:>3}")
            if r["error"]:
                line += f"  err='{r['error'][:60]}'"
            print(line)
        n_ok   = (df_report["status"] == "ok").sum()
        n_fail = (df_report["status"].astype(str).str.startswith("error")).sum()
        print("-" * 72)
        print(f"Summary: {n_ok}/{len(df_report)} patients fully processed   "
              f"{n_fail} failed   "
              f"(total Unknown dropped: {df_report['n_channels_unknown_dropped'].sum()};   "
              f"total WM used: {df_report['n_wm_used'].sum()})")

        # Persist (TSV; survives kernel restart and is reviewable in Excel/etc.)
        try:
            out_tsv = os.path.join(run_root_raw, "wm_reref_report.tsv")
            os.makedirs(os.path.dirname(out_tsv) or ".", exist_ok=True)
            df_report.to_csv(out_tsv, sep="\t", index=False)
            print(f"  saved -> {out_tsv}")
        except Exception as _e_save:
            print(f"  [warn] could not save report TSV: {_e_save}")



EL036
[LF 21:25:11] Patient: EL036 | Block: LM
[LF 21:25:11] Raw dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL036\task_FBM\data_LM\raw
[LF 21:25:11] Prep dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL036\task_FBM\data_LM\prep0
  [EL036] dropped 10 'Unknown' channels (no parcellation in TSV)
[error] EL036: WM reref failed — [reref] EL036: WM channels were found but none were used after exclusions — cannot guarantee WM rereferencing. Check bad_channels_manual and available WM channels.


EL038
[LF 21:25:18] Patient: EL038 | Block: LM
[LF 21:25:18] Raw dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL038\task_FBM\data_LM\raw
[LF 21:25:18] Prep dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL038\task_FBM\data_LM\prep0
  [EL038] dropped 6 'Unknown' channels (no parcellation in TSV)
  audio: *************************************************************************** done
  reading: *

In [ ]:
# === EL036 WM exclusion diagnosis ===
pid = "EL036"
_, _, names, _, _, _wm_tsv = _load_signals_and_prep_for_patient(pid)

import re as _re
if pid in getattr(cfg, "STRIP_HEMI_PATIENTS", set()):
    names = [_re.sub(r"_(?:[LR])(?=\d)", "", str(n)) for n in names]

wm_tsv_raw = io.derive_wm_channels_from_electrodes_tsv(
    _wm_tsv or io.electrodes_tsv_path_for_patient(pid))
wm_idx     = io.wm_indices_for_patient(pid, names, electrodes_tsv_pattern=_wm_tsv)
wm_in_sig  = [names[i] for i in wm_idx]
bad_raw    = cfg.bad_channels_manual.get(pid, [])
used       = [n for n in wm_in_sig if n not in set(bad_raw)]

print(f"WM in TSV   ({len(wm_tsv_raw)}): {wm_tsv_raw}")
print(f"WM in signal({len(wm_in_sig)}): {wm_in_sig}")
print(f"bad_manual  ({len(bad_raw)}): {bad_raw}")
print(f"WM usable   ({len(used)}): {used}")
print(f"min_wm=3 → {'PASS' if len(used) >= 3 else 'FAIL'}")

In [ ]:
# ============================================================
# WM Rereferencing Debug Cell
# Identifies patients that failed WM reref in the last run and
# diagnoses why by comparing channel names between the electrodes
# TSV (WM source) and the loaded signal channel names.
# ============================================================

import glob, os
import numpy as np
import pandas as pd

# ---- 1. Summary list of WM failures from the last run ----
# Re-run the WM check per patient without running ERSP
print("=" * 60)
print("WM REREFERENCING FAILURE REPORT")
print("=" * 60)

wm_failures = []  # list of (pid_raw, patient_id, reason, detail)

for pid_raw in cfg.patient_ids:
    pid_str = str(pid_raw)
    try:
        patient_id, signals, names, fs, prep_dir, _wm_tsv = _load_signals_and_prep_for_patient(pid_raw)
    except Exception as e:
        print(f"  [skip] {pid_raw}: could not load — {e}")
        continue

    # Resolve the electrodes TSV pattern
    if _wm_tsv:
        tsv_pattern = _wm_tsv
    else:
        tsv_pattern = io.electrodes_tsv_path_for_patient(patient_id)

    # Get WM labels from TSV
    wm_from_tsv_raw = []
    try:
        tsv_files = glob.glob(tsv_pattern)
        if tsv_files:
            df_elec = pd.read_csv(tsv_files[0], sep="\t")
            for _, row in df_elec.iterrows():
                try:
                    w1 = float(row.get("tissueWeights_1", float("nan")))
                    tokens = str(row.get("tissueLabel", "")).strip().split()
                    if (w1 > 0.97) and tokens and tokens[0].startswith("wm-"):
                        wm_from_tsv_raw.append(str(row.get("name", "")))
                except Exception:
                    pass
        else:
            wm_from_tsv_raw = None  # TSV not found
    except Exception as e:
        wm_from_tsv_raw = None

    # Get WM indices using the actual function
    try:
        wm_idx = io.wm_indices_for_patient(patient_id, names, electrodes_tsv_pattern=_wm_tsv)
    except Exception as e:
        wm_idx = []

    bad_manual = getattr(cfg, "bad_channels_manual", {}).get(patient_id, [])
    bad_norm = set(io.normalize_label(b) for b in bad_manual)
    wm_names_matched = [names[i] for i in wm_idx]
    wm_after_exclusions = [n for n in wm_names_matched if io.normalize_label(n) not in bad_norm]

    if wm_from_tsv_raw is None:
        reason = "TSV_NOT_FOUND"
        wm_failures.append((pid_raw, patient_id, reason, ""))
    elif not wm_from_tsv_raw:
        reason = "NO_WM_IN_TSV"
        wm_failures.append((pid_raw, patient_id, reason, ""))
    elif not wm_idx:
        reason = "NO_NAME_MATCH"
        wm_failures.append((pid_raw, patient_id, reason, f"TSV WM names: {wm_from_tsv_raw}"))
    elif not wm_after_exclusions:
        reason = "ALL_EXCLUDED_BY_BAD_CHANNELS"
        wm_failures.append((pid_raw, patient_id, reason, f"WM matched: {wm_names_matched}, bad_manual: {bad_manual}"))
    else:
        pass  # success

print(f"\nPatients that FAILED WM rereferencing ({len(wm_failures)} total):")
print(f"{'Patient raw':<15} {'Patient ID':<15} {'Reason':<35} Details")
print("-" * 100)
for pid_raw, patient_id, reason, detail in wm_failures:
    print(f"  {str(pid_raw):<13} {patient_id:<15} {reason:<35} {detail}")

# ---- 2. Deep name-mismatch diagnosis for NO_NAME_MATCH cases ----
no_match = [(pr, pi, d) for pr, pi, reason, d in wm_failures if reason == "NO_NAME_MATCH"]
if no_match:
    print("\n" + "=" * 60)
    print("NAME MISMATCH DIAGNOSIS")
    print("=" * 60)
    for pid_raw, patient_id, _ in no_match:
        print(f"\n--- {pid_raw} / {patient_id} ---")
        try:
            patient_id2, signals, names, fs, prep_dir, _wm_tsv = _load_signals_and_prep_for_patient(pid_raw)
        except Exception as e:
            print(f"  Could not load: {e}"); continue

        tsv_pattern = _wm_tsv or io.electrodes_tsv_path_for_patient(patient_id)
        tsv_files = glob.glob(tsv_pattern)
        if not tsv_files:
            print(f"  No TSV found at: {tsv_pattern}"); continue

        df_elec = pd.read_csv(tsv_files[0], sep="\t")
        all_tsv_names = df_elec["name"].tolist() if "name" in df_elec.columns else []
        wm_tsv_names = []
        for _, row in df_elec.iterrows():
            try:
                w1 = float(row.get("tissueWeights_1", float("nan")))
                tokens = str(row.get("tissueLabel", "")).strip().split()
                if (w1 > 0.97) and tokens and tokens[0].startswith("wm-"):
                    wm_tsv_names.append(str(row.get("name", "")))
            except Exception:
                pass

        print(f"  Signal channels (first 20): {names[:20]}")
        print(f"  Signal channels normalized: {[io.normalize_label(n) for n in names[:20]]}")
        print(f"  WM channels in TSV (raw):   {wm_tsv_names}")
        print(f"  WM channels normalized:     {[io.normalize_label(n) for n in wm_tsv_names]}")

        # Show which normalized WM names are missing from signals
        sig_norm = set(io.normalize_label(n) for n in names)
        for wm_raw in wm_tsv_names:
            wm_norm = io.normalize_label(wm_raw)
            if wm_norm in sig_norm:
                matched = [n for n in names if io.normalize_label(n) == wm_norm]
                print(f"  [OK]    TSV '{wm_raw}' -> '{wm_norm}'  matches signal: {matched}")
            else:
                # Find closest signal names for diagnosis
                close = [n for n in names if wm_norm[:4] in io.normalize_label(n) or io.normalize_label(n)[:4] in wm_norm][:5]
                print(f"  [MISS]  TSV '{wm_raw}' -> '{wm_norm}'  NOT found. Closest signals: {close}")

# ---- 3. Show bad_channels_manual exclusion detail for ALL_EXCLUDED cases ----
all_excl = [(pr, pi, d) for pr, pi, reason, d in wm_failures if reason == "ALL_EXCLUDED_BY_BAD_CHANNELS"]
if all_excl:
    print("\n" + "=" * 60)
    print("BAD_CHANNELS_MANUAL EXCLUSION DETAIL")
    print("=" * 60)
    for pid_raw, patient_id, detail in all_excl:
        print(f"\n--- {pid_raw} / {patient_id} ---")
        print(f"  {detail}")
        bad_manual = getattr(cfg, "bad_channels_manual", {}).get(patient_id, [])
        if not bad_manual:
            print("  (no bad_channels_manual entry — exclusion came from apply_wm_reference_with_exclusions)")
        else:
            print(f"  bad_channels_manual has {len(bad_manual)} entries: {bad_manual}")
            print("  → All WM channels happen to be listed as bad. Remove them from bad_channels_manual to fix.")
